# **本轮参数**
* Model: ResNet18 (pretrained)
* Training: Full fine-tuning
* Batch size: 32
* Learning rate: 1e-4
* Weight decay: 1e-4
* Epochs: 5
* Image size: 224×224
* Seed: 42

# **🛠️ STEP A 数据准备**

## **1. 基础环境 + Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## **2. 基础配置**

In [ ]:
import os
import random
import numpy as np
import torch

# ===== Seed =====
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# ===== 路径 =====
DATA_ROOT = "/content/drive/MyDrive/MLP/Project5/dataset_project5"

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VAL_DIR   = os.path.join(DATA_ROOT, "val")
TEST_DIR  = os.path.join(DATA_ROOT, "test")

CKPT_DIR = "/content/drive/MyDrive/MLP/Project5/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

LATEST_CKPT = os.path.join(CKPT_DIR, "resnet18_baseline_latest.pth")
BEST_CKPT   = os.path.join(CKPT_DIR, "resnet18_baseline_best.pth")

# ===== 训练参数 =====
BATCH_SIZE = 32
IMG_SIZE = 224
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 5
NUM_WORKERS = 0   # 先稳住

# ===== 设备 =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## **3. 数据加载**

In [ ]:
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = ImageFolder(TRAIN_DIR, transform=transform)
val_dataset   = ImageFolder(VAL_DIR, transform=transform)

print("Class mapping:", train_dataset.class_to_idx)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# **🛠️ STEP B 1.1 全量 fine-tune 训练 Baseline 模型**

## **1. 模型（全量 fine-tune）**

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# 全量 fine-tune
for param in model.parameters():
    param.requires_grad = True

# 改 fc
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

print(model.fc)

## **2. checkpoint**
防止训练中断数据丢失，每轮保存checkpoint

In [ ]:
def save_checkpoint(epoch, model, optimizer, train_loss, val_acc, best_val_acc, path):
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_acc": val_acc,
        "best_val_acc": best_val_acc
    }, path)

def load_checkpoint(model, optimizer, path):
    if os.path.exists(path):
        ckpt = torch.load(path)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        print("Resumed from checkpoint")
        return ckpt["epoch"] + 1, ckpt["best_val_acc"]
    else:
        print("Start from scratch")
        return 0, 0.0

## **3. 训练 + 验证函数**

In [ ]:
import time
from sklearn.metrics import accuracy_score

def train_one_epoch():
    model.train()
    total_loss = 0
    total = 0

    start = time.time()

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        total += images.size(0)

        if (i+1) % 50 == 0:
            print(f"Step {i+1}/{len(train_loader)} Loss {loss.item():.4f}")

    return total_loss / total, time.time() - start


def evaluate():
    model.eval()
    preds, labels_all = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            pred = torch.argmax(outputs, dim=1).cpu().numpy()

            preds.extend(pred)
            labels_all.extend(labels.numpy())

    return accuracy_score(labels_all, preds)

## **4. 正式训练（5 epoch）**
数据量不大，epoch太多了容易过拟合

In [ ]:
start_epoch, best_val_acc = load_checkpoint(model, optimizer, LATEST_CKPT)

for epoch in range(start_epoch, NUM_EPOCHS):
    print(f"\n====== Epoch {epoch+1}/{NUM_EPOCHS} ======")

    train_loss, t = train_one_epoch()
    val_acc = evaluate()

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Acc: {val_acc:.4f}")
    print(f"Time: {t/60:.2f} min")

    # 保存 latest
    save_checkpoint(epoch, model, optimizer, train_loss, val_acc, best_val_acc, LATEST_CKPT)

    # 保存 best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(epoch, model, optimizer, train_loss, val_acc, best_val_acc, BEST_CKPT)
        print("✅ Best model updated")



---



# **🛠️ STEP B 1.2 Test 集 + JPEG 退化集测试 Baseline 模型**

## **1. 先准备 test 和 JPEG test 的 DataLoader**

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

JPEG_TEST_DIR = os.path.join(DATA_ROOT, "test_jpeg_q30")

test_dataset = ImageFolder(TEST_DIR, transform=transform)
jpeg_test_dataset = ImageFolder(JPEG_TEST_DIR, transform=transform)

print("Test size:", len(test_dataset))
print("JPEG Test size:", len(jpeg_test_dataset))
print("Class mapping (test):", test_dataset.class_to_idx)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

jpeg_test_loader = DataLoader(
    jpeg_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

## **2.重新加载 best checkpoint**

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# 重新建立同样结构的模型
eval_model = models.resnet18(weights=None)
eval_model.fc = nn.Linear(eval_model.fc.in_features, 2)
eval_model = eval_model.to(device)

# 读取 best checkpoint
ckpt = torch.load(BEST_CKPT, map_location=device)
eval_model.load_state_dict(ckpt["model"])

eval_model.eval()
print("Loaded best checkpoint from:", BEST_CKPT)
print("Best val acc stored in checkpoint:", ckpt.get("best_val_acc", "N/A"))

## **3. 定义“跑一整个测试集”的函数**

定义一个通用函数，它能：

跑完整个 loader，收集：真实标签(y_true)，预测标签 (y_pred)，fake 概率 (p_fake)

In [ ]:
import numpy as np
import torch.nn.functional as F

def run_inference(model, loader, device):
    model.eval()

    y_true = []
    y_pred = []
    p_fake = []   # fake 的概率（因为 fake=0）

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            outputs = model(images)
            probs = F.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)

            y_true.extend(labels.numpy().tolist())
            y_pred.extend(preds.tolist())
            p_fake.extend(probs[:, 0].tolist())   # 因为 fake=0

    return np.array(y_true), np.array(y_pred), np.array(p_fake)

## **4. 在 original test 和 JPEG test 上真正跑预测**
把同一个模型分别拿去测：
原始 test vs JPEG test

In [ ]:
y_true_orig, y_pred_orig, p_fake_orig = run_inference(eval_model, test_loader, device)
y_true_jpeg, y_pred_jpeg, p_fake_jpeg = run_inference(eval_model, jpeg_test_loader, device)

print("Original test samples:", len(y_true_orig))
print("JPEG test samples:", len(y_true_jpeg))

# **🛠️STEP C 指标评估**

把预测结果变成真正能汇报的数字：Accuracy/Precision/Recall/F1

*特别注意 *pos_label* —— 因为现在的数据映射是：fake = 0 / real = 1

我们项目更关心的是 fake detection，所以计算 precision/recall/f1 时，要把：
fake 设为 positive class

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(y_true, y_pred, pos_label=0):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=pos_label)
    rec = recall_score(y_true, y_pred, pos_label=pos_label)
    f1 = f1_score(y_true, y_pred, pos_label=pos_label)
    return acc, prec, rec, f1

orig_acc, orig_prec, orig_rec, orig_f1 = compute_metrics(y_true_orig, y_pred_orig, pos_label=0)
jpeg_acc, jpeg_prec, jpeg_rec, jpeg_f1 = compute_metrics(y_true_jpeg, y_pred_jpeg, pos_label=0)

print("=== Original Test ===")
print(f"Accuracy : {orig_acc:.4f}")
print(f"Precision: {orig_prec:.4f}")
print(f"Recall   : {orig_rec:.4f}")
print(f"F1       : {orig_f1:.4f}")

print("\n=== JPEG Test ===")
print(f"Accuracy : {jpeg_acc:.4f}")
print(f"Precision: {jpeg_prec:.4f}")
print(f"Recall   : {jpeg_rec:.4f}")
print(f"F1       : {jpeg_f1:.4f}")



---



# **🛠️ STEP D 1.1 XAI 输出前置准备**

## **1. 定义带 path 的 inference 函数**

之前已经有：Accuracy / Precision / Recall / F1 4个指标，但这些只是“整体统计结果”，XAI 需要的是“每一张图片的预测结果”，所以要让模型在预测时记住每张图是谁(path)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

def run_inference_with_path(model, dataset, loader, device):
    model.eval()

    y_true = []
    y_pred = []
    p_fake = []
    paths = []

    ptr = 0  # 指向 dataset.samples 的当前位置

    with torch.no_grad():
        for images, labels in loader:
            batch_size = len(labels)
            images = images.to(device)

            outputs = model(images)
            probs = F.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)

            batch_paths = dataset.samples[ptr:ptr + batch_size]
            ptr += batch_size

            for j in range(batch_size):
                y_true.append(labels[j].item())
                y_pred.append(int(preds[j]))
                p_fake.append(float(probs[j][0]))   # fake=0
                paths.append(batch_paths[j][0])

    return pd.DataFrame({
        "path": paths,
        "true_label": y_true,
        "pred_label": y_pred,
        "p_fake": p_fake
    })

## **2. 重新生成 current baseline 的 predictions**

In [ ]:
OUT_DIR = "/content/drive/MyDrive/MLP/Project5/baseline_eval_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

df_orig = run_inference_with_path(eval_model, test_dataset, test_loader, device)
df_jpeg = run_inference_with_path(eval_model, jpeg_test_dataset, jpeg_test_loader, device)

df_orig.to_csv(os.path.join(OUT_DIR, "predictions_original_with_path.csv"), index=False)
df_jpeg.to_csv(os.path.join(OUT_DIR, "predictions_jpeg_with_path.csv"), index=False)

print("Saved:")
print(os.path.join(OUT_DIR, "predictions_original_with_path.csv"))
print(os.path.join(OUT_DIR, "predictions_jpeg_with_path.csv"))
print(df_orig.head())
print(df_jpeg.head())

## **3. 挑 JPEG 下的 FN**
FN: 本来是fake，模型认为是real的被误判断图片

In [ ]:
fn_jpeg = df_jpeg[(df_jpeg["true_label"] == 0) & (df_jpeg["pred_label"] == 1)]
print("FN in JPEG:", len(fn_jpeg))

fn_jpeg_sorted = fn_jpeg.sort_values(by="p_fake")  # 越小表示模型越不相信它是fake
fn_samples = fn_jpeg_sorted.head(3)
fn_samples

## **4. 用Grad-CAM 对这 3 张 JPEG 下的 FN 样本画热力图**

In [ ]:
# 下载grad-cam
!pip -q install grad-cam

In [ ]:
# 导入包
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn.functional as F

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [ ]:
# 准备和训练时一致的 transform
from torchvision import transforms

IMG_SIZE = 224

cam_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# 建立 Grad-CAM 对象 (ResNet18 一般取最后一个残差块作为 target layer)
target_layers = [eval_model.layer4[-1]]
cam = GradCAM(model=eval_model, target_layers=target_layers)

In [ ]:
# 写一个可视化函数来：读图片 + 跑模型预测 + 生成热力图并显示

def visualize_gradcam(img_path, model, cam, device, transform, class_names=None):
    # 1) 读取原图
    img = Image.open(img_path).convert("RGB")
    img = img.resize((224, 224))
    img_np = np.array(img).astype(np.float32) / 255.0

    # 2) 转成模型输入
    input_tensor = transform(img).unsqueeze(0).to(device)

    # 3) 先看模型预测
    model.eval()
    with torch.no_grad():
        outputs = model(input_tensor)
        probs = F.softmax(outputs, dim=1).cpu().numpy()[0]
        pred_class = int(np.argmax(probs))

    # 4) 以“模型预测的类别”作为解释目标
    targets = [ClassifierOutputTarget(pred_class)]

    # 5) 生成 CAM
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    # 6) 叠加到原图
    visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

    # 7) 显示
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img_np)
    plt.axis("off")
    plt.title("Original Image")

    plt.subplot(1, 2, 2)
    plt.imshow(visualization)
    plt.axis("off")
    if class_names is None:
        plt.title(f"Grad-CAM | pred={pred_class} | probs={probs}")
    else:
        plt.title(f"Grad-CAM | pred={class_names[pred_class]} | probs={np.round(probs, 4)}")

    plt.tight_layout()
    plt.show()

    return pred_class, probs

In [ ]:
# 对选出来的 3 张 FN 样本画图

class_names = ["fake", "real"]

for path in fn_samples["path"].tolist():
    print("Path:", path)
    pred_class, probs = visualize_gradcam(
        img_path=path,
        model=eval_model,
        cam=cam,
        device=device,
        transform=cam_transform,
        class_names=class_names
    )

# **🛠️ STEP D 1.2 风险因素分析**